In [ ]:
import gzip
import random
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np

In [12]:
def readGz(path):
    for l in gzip.open(path, "rt"):
        yield eval(l)


def readCSV(path):
    f = gzip.open(path, "rt")
    f.readline()
    for l in f:
        yield l.strip().split(",")

In [13]:
allRatings = [(u, b, int(r)) for u, b, r in readCSV("train_Interactions.csv.gz")]
ratingsTrain = allRatings[:190000]
ratingsValid = allRatings[190000:]
ratingsPerUser = defaultdict(list)
ratingsPerItem = defaultdict(list)

for u, b, r in ratingsTrain:
    ratingsPerUser[u].append((b, r))
    ratingsPerItem[b].append((u, r))

In [ ]:
def create_validation_set(ratingsValid):
    allBooks = list(set(b for _, b, _ in allRatings))
    readValid = [(u, b, 1) for u, b, _ in ratingsValid]
    notRead = []
    for u, _, _ in readValid:
        while True:
            b_neg = random.choice(allBooks)
            if b_neg not in userBooks[u]:
                notRead.append((u, b_neg, 0))
                break
    return readValid + notRead

In [15]:
totalRead = 0
bookCount = defaultdict(int)
userCount = defaultdict(int)
for user,book,_ in ratingsTrain:
    bookCount[book] += 1
    userCount[user] += 1
    totalRead += 1

In [16]:
def build_popular_books(bookCount, totalRead, threshold=0.72):
    mostPopular = [(bookCount[x], x) for x in bookCount]
    mostPopular.sort()
    mostPopular.reverse()
    
    count = 0
    popular = set()
    for ic, i in mostPopular:
        count += ic
        popular.add(i)
        if count > totalRead * threshold:
            break
    return popular

def build_active_users(userCount, totalRead, threshold=0.01):
    mostActive = [(userCount[x], x) for x in userCount]
    mostActive.sort()
    mostActive.reverse()
    
    count = 0
    active = set()
    for ic, i in mostActive:
        count += ic
        active.add(i)
        if count > totalRead * threshold:
            break
    return active

In [17]:
userBooks = {u: set(b for b, _ in bs) for u, bs in ratingsPerUser.items()}
bookUsers = {b: set(u for u, _ in us) for b, us in ratingsPerItem.items()}
popularBooks = build_popular_books(bookCount, totalRead)
activeUsers = build_active_users(userCount, totalRead)

def Jaccard(s1, s2):
    return len(set(s1) & set(s2)) / len(set(s1) | set(s2))

def predictRead(u, b):
    if u not in userBooks or not userBooks[u]:
        return int(b in popularBooks)

    if b not in bookUsers or not bookUsers[b]:
        return int(u in activeUsers)
    
    maxSim = 0
    for bp in userBooks[u]:
        if bp in bookUsers:
            sim = Jaccard(bookUsers[b], bookUsers[bp])
            maxSim = max(maxSim, sim)
            if maxSim > 0.025:
                return 1
    
    return int(b in popularBooks)

In [ ]:
predictions = open("predictions_Read.csv", "w")
for l in tqdm(open("pairs_Read.csv"), desc="Writing read predictions"):
    if l.startswith("userID"):
        predictions.write(l)
        continue
    u, b = l.strip().split(",")
    predictions.write(f"{u},{b},{predictRead(u, b)}\n")
predictions.close()

Writing read predictions: 20001it [00:02, 7114.46it/s]


In [ ]:
# Create validation set
userBooks_train = {u: set(b for b, _ in bs) for u, bs in ratingsPerUser.items()}
validation_set = create_validation_set(ratingsValid, userBooks_train)

# Get predictions and actual labels
y_true = [label for _, _, label in validation_set]
y_pred = [predictRead(u, b) for u, b, _ in validation_set]

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

# Calculate global average rating
global_avg_rating = np.mean([r for _, _, r in allRatings])
print(f"Global Average Rating: {global_avg_rating:.4f}")

# Calculate confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Calculate average rating per book (across all users)
book_avg_ratings = defaultdict(list)
for u, b, r in allRatings:
    book_avg_ratings[b].append(r)

book_avg = {b: np.mean(ratings) for b, ratings in book_avg_ratings.items()}

# Calculate average rating for each confusion matrix cell
# Group books by their confusion matrix cell
cell_books = {
    (0, 0): [],  # True Negative
    (0, 1): [],  # False Positive
    (1, 0): [],  # False Negative
    (1, 1): []   # True Positive
}

for (u, b, true_label), pred_label in zip(validation_set, y_pred):
    cell_books[(true_label, pred_label)].append(b)

# Calculate average ratings for each cell
avg_ratings = {}
for cell, books in cell_books.items():
    if len(books) > 0:
        # Get average rating for each book, then average across books
        book_ratings = [book_avg.get(b, 0.0) for b in books if b in book_avg]
        if len(book_ratings) > 0:
            avg_ratings[cell] = np.mean(book_ratings)
        else:
            avg_ratings[cell] = 0.0
    else:
        avg_ratings[cell] = 0.0

# Plot confusion matrix with annotations
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'}, 
            xticklabels=['Not Read (0)', 'Read (1)'], 
            yticklabels=['Not Read (0)', 'Read (1)'])

# Add average ratings to each cell
for i in range(2):
    for j in range(2):
        count = cm[i, j]
        avg_rating = avg_ratings.get((i, j), 0.0)
        plt.text(j + 0.5, i + 0.7, f'\nAvg Rating: {avg_rating:.3f}', 
                ha='center', va='top', fontsize=10, color='red', weight='bold')

plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.title(f'Confusion Matrix\nAccuracy: {accuracy:.4f} | Global Avg Rating: {global_avg_rating:.4f}', 
          fontsize=14, weight='bold')
plt.tight_layout()
plt.show()


KeyError: 'u96558248'